# ARC Prize 2026 — Evaluacion (kernel GPU, offline)

Carga Qwen2.5 base + adaptador LoRA entrenado y mide **pass@2** en un subconjunto de
`evaluation` con inferencia **AIRV** (sin TTT). Adjuntar: codigo (`arc-agi2-code`),
competencia, modelo base (`qwen2.5/.../7b-instruct`) y el adaptador (`arc-agi2-model`).

In [ ]:
import os, sys, glob, json
EVAL_LIMIT = 40      # nº de tareas de evaluation (subconjunto para ahorrar quota)
N_AUG = 4            # vistas AIRV por test input
MAX_NEW_TOKENS = 1024

print('INPUTS:', glob.glob('/kaggle/input/*'))
arc_init = glob.glob('/kaggle/input/**/src/arc/__init__.py', recursive=True)
assert arc_init, 'falta dataset de codigo'
SRC_DIR = os.path.dirname(os.path.dirname(arc_init[0])); sys.path.insert(0, SRC_DIR)
ch = glob.glob('/kaggle/input/**/arc-agi_evaluation_challenges.json', recursive=True)
assert ch, 'faltan datos de la competencia'
COMP_DIR = os.path.dirname(ch[0])

# base model (config.json + pesos, sin adapter) y adaptador LoRA (adapter_config.json)
MODEL_DIR = None
for cfg in glob.glob('/kaggle/input/**/config.json', recursive=True):
    d = os.path.dirname(cfg); fs = os.listdir(d)
    if any(f.endswith(('.safetensors', '.bin')) for f in fs) and 'adapter_config.json' not in fs:
        MODEL_DIR = d; break
ad = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
LORA_DIR = os.path.dirname(ad[0]) if ad else None
assert MODEL_DIR and LORA_DIR, f'MODEL_DIR={MODEL_DIR} LORA_DIR={LORA_DIR}'
print('MODEL_DIR =', MODEL_DIR); print('LORA_DIR =', LORA_DIR); print('COMP_DIR =', COMP_DIR)

In [ ]:
import torch
from arc.model import load_model, generate_grid
from peft import PeftModel

cap = torch.cuda.get_device_capability(0)[0]
model, tok = load_model(MODEL_DIR, device_map='balanced',
                        dtype='bfloat16' if cap >= 8 else 'float16')
model = PeftModel.from_pretrained(model, LORA_DIR)
model.eval()
print('GPU:', torch.cuda.get_device_name(0), '| n_gpus', torch.cuda.device_count())
predict_fn = lambda prompt: generate_grid(model, tok, prompt,
                                          max_new_tokens=MAX_NEW_TOKENS, constrained=True)

In [ ]:
import time
from arc.data import load_split
from arc.infer import predict_task
from arc.metrics import score_submission

tasks = dict(list(load_split('evaluation', data_dir=COMP_DIR).items())[:EVAL_LIMIT])
solutions = {t: [p.output for p in task.test] for t, task in tasks.items()}
submission, t0 = {}, time.time()
for i, (tid, task) in enumerate(tasks.items()):
    submission[tid] = predict_task(task, predict_fn, n_aug=N_AUG)
    if (i+1) % 5 == 0:
        sc = score_submission({k: submission[k] for k in list(submission)},
                              {k: solutions[k] for k in submission})
        print(f'{i+1}/{len(tasks)} | score parcial {sc:.3f} | {time.time()-t0:.0f}s')
score = score_submission(submission, solutions)
print(f'\n=== SCORE pass@2 en {len(tasks)} tareas de evaluation: {score:.4f} ===')
json.dump({'score': score, 'n': len(tasks), 'n_aug': N_AUG}, open('/kaggle/working/eval_score.json','w'))